<a href="https://colab.research.google.com/github/Firojpaudel/RAGDocs/blob/main/Pdf_answering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -Uqq llama-index llama-index-llms-huggingface transformers torch pypdf gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!pip install -Uqq llama-index-embeddings-huggingface sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 340.6/340.6 kB 11.3 MB/s eta 0:00:00


In [3]:
from google.colab import userdata
token = userdata.get('HF_TOKEN')

In [4]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext, load_index_from_storage
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from huggingface_hub import login
from google.colab import files, userdata
import torch
import os


login(token)

# Upload PDF
print("Please upload your PDF:")
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

# Load the PDF
documents = SimpleDirectoryReader(input_files=[pdf_path]).load_data()

# Set up Mistral-7B with optimized offloading
llm = HuggingFaceLLM(
    model_name="mistralai/Mistral-7B-Instruct-v0.1",
    tokenizer_name="mistralai/Mistral-7B-Instruct-v0.1",
    device_map="auto",
    model_kwargs={
        "torch_dtype": torch.float16,
        "token": token,
        "offload_buffers": True,  # Optimize memory
    },
    max_new_tokens=256,  # Limit output size
)

# Set up a local embedding model
embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create and save index to disk
index_dir = "./index_storage"
if not os.path.exists(index_dir):
    os.makedirs(index_dir)
index = VectorStoreIndex.from_documents(documents, llm=llm, embed_model=embed_model)
index.storage_context.persist(persist_dir=index_dir)

# Free up memory by deleting the in-memory index
del index
torch.cuda.empty_cache()

# CLI query loop
print("\nPDF loaded and indexed to disk. You can now ask questions!")
while True:
    question = input("Ask a question about the PDF (or type 'exit' to quit): ")
    if question.lower() == 'exit':
        print("Exiting...")
        del llm
        torch.cuda.empty_cache()
        break

    # Load index from disk
    storage_context = StorageContext.from_defaults(persist_dir=index_dir)
    index = load_index_from_storage(storage_context, llm=llm, embed_model=embed_model)

    # Query
    query_engine = index.as_query_engine(llm=llm)
    response = query_engine.query(question)
    print(f"Answer: {response}\n")

    # Clean up memory after query
    del index
    torch.cuda.empty_cache()

# Final cleanup
print("Cleaning up...")
del llm
torch.cuda.empty_cache()

Please upload your PDF:


Saving SAD_Project__Finalized.pdf to SAD_Project__Finalized.pdf


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Answer: 

GEM is an architecture from prior research that was leveraged in this project to ensure robust performance on niche datasets. It is mentioned in the context information as a part of the system design and results section.

Ask a question about the PDF (or type 'exit' to quit): and which dataset plus model was it finetuned on?


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Answer: 
The model was fine-tuned on the Bitext customer support llm chatbot training dataset using the BART-base model.

Ask a question about the PDF (or type 'exit' to quit): exit
Exiting...
Cleaning up...


NameError: name 'llm' is not defined